## AgentCore Runtime의 End-to-End Stateless MCP Client

이 예제에서는 AgentCore Runtime에 배포된 전체 MCP client capability(MCP Spec 기반)를 보여줍니다.

예를 들어 MCP native elicitation과 sampling을 활용할 수 있는 MCP server를 AgentCore Runtime에 생성할 때 유용합니다.

이 튜토리얼에서는 다음 내용을 학습합니다.

* elicitation과 sampling이 포함된 MCP server를 생성하는 방법
* AgentCore Runtime에 배포하는 방법
* 배포된 server를 호출하는 방법

### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                 |
|:--------------------|:----------------------------------------------------------|
| 튜토리얼 유형       | Runtime에 Elicitation 및 Sampling 호스팅                  |
| Tool 유형           | MCP server                                                |
| 튜토리얼 구성 요소  | AgentCore Runtime에 호스팅, MCP server 생성               |
| 튜토리얼 분야       | 산업 공통                                                 |
| 예제 난이도         | 중급                                                       |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 MCP Client          |

### 튜토리얼 아키텍처

이 튜토리얼 Notebook에서는 하나의 agent를 구축합니다. 먼저 네 개의 tool과 함께 agent를 AgentCore Runtime에 배포합니다. 그런 다음 prompt를 추가하도록 업데이트하고 마지막으로 resource를 배포하도록 다시 업데이트합니다.

이제 시작해 보겠습니다.

In [ ]:
!uv pip install -qU -r requirements.txt

script가 helpers 폴더에 액세스할 수 있도록 다음 path를 추가합니다.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

### MCP Server 생성

먼저 Elicitation만 포함된 MCP Server를 생성하고 이후 다른 기능을 추가합니다.

이 MCP는 네 개의 MCP tool로 구성됩니다.

- **add_expense_interactive**: 값을 질문하며 대화형으로 지출 추가

데이터는 DynamoDB table에 영구 저장되므로 이 튜토리얼의 다음 단계에서도 사용할 수 있습니다. 

다음 코드를 실행하여 MCP server 로컬 파일과 requirements 파일을 생성합니다.

In [ ]:
%%writefile agents/mcp_client_features.py
import os
from pydantic import BaseModel
from fastmcp import FastMCP, Context
from fastmcp.server.elicitation import AcceptedElicitation
from dynamo_utils import FinanceDB

mcp = FastMCP(name='ElicitationMCP')

# 모든 AgentCore/Lambda container에 AWS_REGION이 안정적으로 설정됨
_region = os.environ.get('AWS_REGION') or os.environ.get('AWS_DEFAULT_REGION') or 'us-east-1'
db = FinanceDB(region_name=_region)

class AmountInput(BaseModel):
    amount: float

class DescriptionInput(BaseModel):
    description: str

class CategoryInput(BaseModel):
    category: str  # 다음 중 하나: food, transport, bills, entertainment, other

class ConfirmInput(BaseModel):
    confirm: str  # Yes 또는 No

@mcp.tool()
async def add_expense_interactive(user_alias: str, ctx: Context) -> str:
    """Interactively add a new expense using elicitation
    Args:
        user_alias: User identifier
    """
    print(f'Debug this method, user_alias: {user_alias}')
    # 1단계: 금액 질문
    result = await ctx.elicit('How much did you spend?', AmountInput)
    if not isinstance(result, AcceptedElicitation):
       return 'Expense entry cancelled.'
    amount = result.data.amount

    # 2단계: 설명 질문
    result = await ctx.elicit('What was it for?', DescriptionInput)
    if not isinstance(result, AcceptedElicitation):
       return 'Expense entry cancelled.'
    description = result.data.description

    # 3단계: category 선택
    result = await ctx.elicit(
        'Select a category (food, transport, bills, entertainment, other):',
        CategoryInput
    )
    if not isinstance(result, AcceptedElicitation):
       return 'Expense entry cancelled.'
    category = result.data.category

    # 4단계: 저장 전 확인
    confirm_msg = f'Confirm: add expense of ${amount:.2f} for {description} (category: {category})? Reply Yes or No'
    result = await ctx.elicit(confirm_msg, ConfirmInput)
    if not isinstance(result, AcceptedElicitation) or result.data.confirm != 'Yes':
        return 'Expense entry cancelled.'

    return db.add_transaction(user_alias, 'expense', -abs(amount), description, category)

if __name__ == '__main__':
    mcp.run(
        transport="streamable-http",
        host="0.0.0.0",
        port=8000,
        stateless_http=False
    )

In [ ]:
%%writefile agents/requirements.txt
fastmcp>=2.10.0
mcp
bedrock-agentcore

`dynamo_utils` 파일을 agents 폴더로 복사합니다.

In [ ]:
!cp ../helpers/dynamo_utils.py agents/dynamo_utils.py

MCP에서 사용할 DynamoDB table이 아직 없으면 생성합니다.

In [ ]:
import boto3

from helpers.dynamo_utils import FinanceDB

region = boto3.session.Session().region_name
db = FinanceDB(region_name=region)
result = db.create_table()
print(f"Region: {region}")
print(result)

#### 인증을 위한 Cognito User Pool 생성

MCP Server에서 인증을 보장하도록 Cognito user pool을 생성합니다.

In [ ]:
from helpers.utils import get_or_create_cognito_pool, reauthenticate_user

print("Setting up Amazon Cognito user pool...")
cognito_config = get_or_create_cognito_pool()  # 이 output cell에서 bearer token을 가져옴
print("Cognito setup completed ✓")

AgentCore execution role을 생성합니다.

In [ ]:
from helpers.utils import create_agentcore_runtime_execution_role, SAMPLE_ROLE_NAME

execution_role_arn_mcp = create_agentcore_runtime_execution_role(SAMPLE_ROLE_NAME)

#### AgentCore Runtime에서 MCP Server 구성 및 시작

AgentCore에서 elicitation server를 구성하고 시작합니다. server 코드는 기본값인 `stateless_http=False`를 사용합니다. AgentCore가 session을 유지하므로 server는 각 elicitation 단계에서 일시 중지했다가 다시 시작할 수 있습니다.

In [ ]:
import json
import boto3
from boto3.session import Session
from bedrock_agentcore_starter_toolkit import Runtime

boto_session = Session()
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]
region = boto_session.region_name

aws_agent_name = "mcp_client_features"
runtime = Runtime()

response = runtime.configure(
    entrypoint="agents/mcp_client_features.py",
    execution_role=execution_role_arn_mcp,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=aws_agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
    protocol="MCP",
    deployment_type="direct_code_deploy",
    runtime_type="PYTHON_3_13",
)
print("Configuration completed:", response)

In [ ]:
launch_result = runtime.launch()
print("Launch completed:", launch_result.agent_arn)

In [ ]:
status_response = runtime.status()
status = status_response.endpoint["status"]
print(f"Final status: {status}")

### Elicitation 테스트

배포된 server를 호출하고 elicitation 작동을 확인합니다.

#### Scenario

점심 식사 지출을 기록합니다. 모든 argument를 tool에 한 번에 전달하는 대신 server가 금액, 용도, category를 차례로 묻고 DynamoDB에 쓰기 전 최종 확인을 요청하며 입력 과정을 단계별로 안내합니다.

이는 실제 사용자용 assistant의 작동 방식과 같습니다. server가 대화를 주도하며 한 번에 하나씩 정보를 수집합니다.

#### 테스트 작동 방식

`fastmcp.Client`는 `elicitation_handler` callback을 지원하는 유일한 MCP client입니다. **server**가 `await ctx.elicit(...)`를 호출할 때마다 열린 session을 통해 client에 `elicitation/create` request를 다시 보냅니다. client는 다음 사전 정의 response를 반환하는 handler를 호출합니다.

Jupyter는 remote server의 `input()` 호출을 대화형으로 받을 수 없으므로 네 가지 답변을 미리 load하여 사용자를 시뮬레이션합니다.

| 단계 | Server 질문 | 응답 |
|:-----|:------------|:-----------|
| 1 | `"How much did you spend?"` | `{"amount": 45.50}` |
| 2 | `"What was it for?"` | `{"description": "Lunch at the office"}` |
| 3 | `"Select a category..."` | `{"category": "food"}` |
| 4 | `"Confirm: add expense of $45.50..."` | `{"confirm": "Yes"}` |

각 response는 key가 해당 단계에서 server가 정의한 Pydantic 모델의 field 이름(`AmountInput`, `DescriptionInput`, `CategoryInput`, `ConfirmInput`)과 일치하는 dict입니다. 마지막 `"Yes"`에서 server가 `db.add_transaction()`을 호출하고 결과를 반환합니다.


In [ ]:
ac_runtime_name = launch_result.agent_id

mcp_url = (
    f"https://bedrock-agentcore.{region}.amazonaws.com"
    f"/runtimes/{ac_runtime_name}/invocations"
    f"?qualifier=DEFAULT&accountId={account_id}"
)

bearer_token = reauthenticate_user(cognito_config.get("client_id"), cognito_config.get("client_secret"))

headers = {
    "authorization": f"Bearer {bearer_token}",
    "Content-Type": "application/json",
}

print(f"MCP URL: {mcp_url}")

In [ ]:
import asyncio
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

# iterator가 항상 새 상태이도록 실행할 때마다 다시 생성
_responses = iter(
    [
        {"amount": 45.50},
        {"description": "Lunch at the office"},
        {"category": "food"},
        {"confirm": "Yes"},
    ]
)


async def elicit_handler(message, response_type, params, context):
    response = next(_responses)
    print(f"  Server asks: {message}")
    print(f"  Auto-responding: {response}\n")
    return response


async def invoke_tool(tool_name: str, tool_params: dict):
    transport = StreamableHttpTransport(url=mcp_url, headers=headers)
    async with Client(transport, elicitation_handler=elicit_handler) as client:
        print("\n    Waiting 2s for session initialization...")
        await asyncio.sleep(2)

        print(f"\n➕ Testing tool: {tool_name}()...")
        try:
            result = await client.call_tool(tool_name, tool_params)
            print(f"   Result: {result.content[0].text}")
        except Exception as e:
            print(f"   Error: {e}")

In [ ]:
tool_name = "add_expense_interactive"
tool_arguments = {"user_alias": "me"}
await invoke_tool(tool_name, tool_arguments)

---

### Sampling을 추가하도록 Runtime 변경

기능을 추가하도록 MCP 코드를 일부 변경합니다.

#### Sampling이란?

표준 MCP에서는 client가 tool을 호출하고 server가 응답합니다. sampling을 사용하면 server가 **tool 실행을 일시 중지**하고 client에 LLM 실행을 요청한 다음 AI 생성 결과를 사용하여 task를 완료할 수 있습니다.

이는 다른 MCP 기능과 다음과 같은 차이가 있습니다.
- **Tools**: client가 argument와 함께 server 호출 → server 응답
- **Prompts**: server가 client에서 LLM으로 전송할 prompt template 반환
- **Resources**: server가 client에서 읽을 수 있는 데이터 노출
- **Elicitation**: server가 client에 **user input** 요청
- **Sampling**: server가 client에 **LLM-generated text** 요청

### 역방향 Flow

```
일반:     Client ──tool call──▶ Server ──result──▶ Client

Sampling: Client ──tool call──▶ Server
                                  │
                                  └──sampling/createMessage──▶ Client
                                  ◀────────LLM response────────┘
                                  │
                                  └──────────result────────────▶ Client
```

server에는 DynamoDB의 **데이터**가 있고 client에는 **LLM**(Bedrock Claude)이 있습니다. Sampling이 둘을 연결합니다.

### Sampling 사용 시점

- tool이 방금 가져온 structured data에서 자연어를 생성해야 할 때
- server 자체에 LLM을 포함하지 않고 AI 기반 분석을 수행하려 할 때
- 서로 다른 client가 다른 LLM을 사용할 수 있어 server를 model-agnostic 상태로 유지할 때

In [ ]:
%%writefile agents/mcp_client_features.py
import os
from pydantic import BaseModel
from fastmcp import FastMCP, Context
from fastmcp.server.elicitation import AcceptedElicitation
from dynamo_utils import FinanceDB

mcp = FastMCP(name='ElicitationMCP')

# 모든 AgentCore/Lambda container에 AWS_REGION이 안정적으로 설정됨
_region = os.environ.get('AWS_REGION') or os.environ.get('AWS_DEFAULT_REGION') or 'us-east-1'
db = FinanceDB(region_name=_region)

class AmountInput(BaseModel):
    amount: float

class DescriptionInput(BaseModel):
    description: str

class CategoryInput(BaseModel):
    category: str  # 다음 중 하나: food, transport, bills, entertainment, other

class ConfirmInput(BaseModel):
    confirm: str  # Yes 또는 No

@mcp.tool()
async def add_expense_interactive(user_alias: str, ctx: Context) -> str:
    """Interactively add a new expense using elicitation
    Args:
        user_alias: User identifier
    """
    print(f'Debug this method, user_alias: {user_alias}')
    # 1단계: 금액 질문
    result = await ctx.elicit('How much did you spend?', AmountInput)
    if not isinstance(result, AcceptedElicitation):
       return 'Expense entry cancelled.'
    amount = result.data.amount

    # 2단계: 설명 질문
    result = await ctx.elicit('What was it for?', DescriptionInput)
    if not isinstance(result, AcceptedElicitation):
       return 'Expense entry cancelled.'
    description = result.data.description

    # 3단계: category 선택
    result = await ctx.elicit(
        'Select a category (food, transport, bills, entertainment, other):',
        CategoryInput
    )
    if not isinstance(result, AcceptedElicitation):
       return 'Expense entry cancelled.'
    category = result.data.category

    # 4단계: 저장 전 확인
    confirm_msg = f'Confirm: add expense of ${amount:.2f} for {description} (category: {category})? Reply Yes or No'
    result = await ctx.elicit(confirm_msg, ConfirmInput)
    if not isinstance(result, AcceptedElicitation) or result.data.confirm != 'Yes':
        return 'Expense entry cancelled.'

    return db.add_transaction(user_alias, 'expense', -abs(amount), description, category)


@mcp.tool()
def add_expense(user_alias: str, amount: float, description: str, category: str = 'other') -> str:
    """Add a new expense transaction.

    Args:
        user_alias: User identifier
        amount: Expense amount (positive number)
        description: Description of the expense
        category: Expense category (food, transport, bills, entertainment, other)
    """
    return db.add_transaction(user_alias, 'expense', -abs(amount), description, category)


@mcp.tool()
async def analyze_spending(user_alias: str, ctx: Context) -> str:
    """Fetch this user's expenses from DynamoDB and use the client's LLM
    to generate a personalised financial analysis.

    Args:
        user_alias: User identifier
    """
    transactions = db.get_transactions(user_alias)
    if not transactions:
        return f'No transactions found for {user_alias}.'

    lines = '\n'.join(
        f"- {t['description']} (${abs(float(t['amount'])):.2f}, {t['category']})"
        for t in transactions
    )

    prompt = (
        f'Here are the recent expenses for a user:\n{lines}\n\n'
        f'Please analyse the spending patterns and give 3 concise, '
        f'actionable recommendations to improve their finances. '
        f'Keep the response under 120 words.'
    )

    ai_analysis = 'Analysis unavailable.'
    try:
        response = await ctx.sample(messages=prompt, max_tokens=300)
        if hasattr(response, 'text') and response.text:
            ai_analysis = response.text
    except Exception:
        pass

    return f'Spending Analysis for {user_alias}:\n\n{ai_analysis}'


if __name__ == '__main__':
    mcp.run(
        transport="streamable-http",
        host="0.0.0.0",
        port=8000,
        stateless_http=False
    )

sampling을 포함하여 다시 배포합니다.

In [ ]:
launch_result = runtime.launch()
print("Launch completed:", launch_result.agent_arn)

#### 새 기능 테스트

MCP server에 방금 추가한 새 기능을 테스트합니다.

In [ ]:
bearer_token = reauthenticate_user(cognito_config.get("client_id"), cognito_config.get("client_secret"))

headers = {
    "authorization": f"Bearer {bearer_token}",
    "Content-Type": "application/json",
}

In [ ]:
import boto3
from mcp.types import CreateMessageResult, TextContent

MODEL_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
bedrock = boto3.client("bedrock-runtime", region_name=region)


def _invoke_bedrock(prompt: str, max_tokens: int) -> str:
    body = json.dumps(
        {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_tokens,
            "messages": [{"role": "user", "content": prompt}],
        }
    )
    resp = bedrock.invoke_model(modelId=MODEL_ID, body=body)
    return json.loads(resp["body"].read())["content"][0]["text"]


async def sampling_handler(messages, params, ctx):
    """서버가 ctx.sample()을 실행할 때 fastmcp.Client가 호출합니다."""
    prompt = (
        messages
        if isinstance(messages, str)
        else " ".join(m.content.text for m in messages if hasattr(m.content, "text"))
    )
    max_tokens = params.maxTokens if params and hasattr(params, "maxTokens") and params.maxTokens else 300
    text = await asyncio.to_thread(_invoke_bedrock, prompt, max_tokens)
    return CreateMessageResult(
        role="assistant",
        content=TextContent(type="text", text=text),
        model=MODEL_ID,
        stopReason="end_turn",
    )

In [ ]:
transport = StreamableHttpTransport(url=mcp_url, headers=headers)

print("Testing Sampling...\n" + "=" * 50)

async with Client(transport, sampling_handler=sampling_handler) as client:
    expenses = [
        {
            "user_alias": "me",
            "amount": 45.50,
            "description": "Dinner",
            "category": "food",
        },
        {
            "user_alias": "me",
            "amount": 120.00,
            "description": "Electricity",
            "category": "bills",
        },
        {
            "user_alias": "me",
            "amount": 15.99,
            "description": "Netflix",
            "category": "entertainment",
        },
        {
            "user_alias": "me",
            "amount": 85.30,
            "description": "Groceries",
            "category": "food",
        },
    ]
    for args in expenses:
        await client.call_tool("add_expense", args)
        print(f"  Added: {args['description']} (${args['amount']:.2f})")

    print("\n  Calling analyze_spending...")
    result = await client.call_tool("analyze_spending", {"user_alias": "me"})

print("=" * 50)
print(f"\n{result.content[0].text}")

---

### 리소스 정리(선택 사항)

아래 셀을 실행하여 이 튜토리얼에서 생성한 모든 AWS 리소스를 삭제합니다.

In [ ]:
from pathlib import Path
from bedrock_agentcore_starter_toolkit.operations.runtime.destroy import (
    destroy_bedrock_agentcore,
)

print("Destroying AgentCore runtime...")
destroy_bedrock_agentcore(config_path=Path(".bedrock_agentcore.yaml"), agent_name=aws_agent_name)

In [ ]:
from helpers.utils import delete_agentcore_runtime_execution_role

# execution role 삭제
print("  🗑️  Deleting Agent execution role...")
delete_agentcore_runtime_execution_role(SAMPLE_ROLE_NAME)
print("  ✅ Execution role deleted")

In [ ]:
from helpers.utils import (
    cleanup_cognito_resources,
    delete_cognito_secret,
    get_cognito_secret,
)

# Cognito와 secret 정리
print("  🗑️  Cleaning up Cognito resources...")
cs = json.loads(get_cognito_secret())
cleanup_cognito_resources(cognito_config.get("pool_id"))
print("  ✅ Cognito resources cleaned up")

print("  🗑️  Deleting customer support secret...")
delete_cognito_secret()
print("  ✅ Customer support secret deleted")

In [ ]:
print("Deleting DynamoDB table...")
result = db.delete_table()
print(result)

In [ ]:
from helpers.utils import local_file_cleanup

print("📁 Starting Local Files cleanup...")
local_file_cleanup()